# Reconstruction Attack
**Goal:** Train a decoder that maps CNN feature embeddings back to X-ray images.
Compare reconstruction quality: baseline (no privacy) vs Differential Privacy (Laplace noise).

**Attack succeeds if:** reconstructed image reveals recognizable anatomical structures.

**Metrics (slide 29):** MSE, PSNR, SSIM, LPIPS, FID

In [ ]:
# ================================
# 1. Setup (same as Untitled3.ipynb)
# ================================
!pip install -q kaggle scikit-image lpips

from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

In [ ]:
# ================================
# 2. Imports
# ================================
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [ ]:
# ================================
# 3. Config
# ================================
base_dir   = "/content/chest_xray"
train_dir  = os.path.join(base_dir, "train")
val_dir    = os.path.join(base_dir, "val")
test_dir   = os.path.join(base_dir, "test")

IMG_SIZE   = 150
BATCH_SIZE = 32
EMBED_DIM  = 36992  # Flatten layer output: 17x17x128

In [ ]:
# ================================
# 4. Same functions as Untitled3.ipynb
#    (copied to keep this notebook self-contained)
# ================================
def add_laplace_noise(image, epsilon=1.0):
    sensitivity = 1.0
    scale = sensitivity / epsilon
    noise = np.random.laplace(loc=0.0, scale=scale, size=image.shape)
    return np.clip(image + noise, 0, 1)

def build_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# ================================
# 5. Data generators
# ================================
normal_datagen = ImageDataGenerator(rescale=1./255)
laplace_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=lambda img: add_laplace_noise(img)
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen_normal = normal_datagen.flow_from_directory(
    train_dir, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE, class_mode='binary'
)
train_gen_laplace = laplace_datagen.flow_from_directory(
    train_dir, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE, class_mode='binary'
)
val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE, class_mode='binary'
)
test_gen = val_datagen.flow_from_directory(
    test_dir, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE, class_mode='binary',
    shuffle=False
)

In [ ]:
# ================================
# 6. Train CNN models
# ================================
tf.keras.backend.clear_session()
model_normal = build_model()
model_normal.fit(train_gen_normal, validation_data=val_gen, epochs=10)

tf.keras.backend.clear_session()
model_laplace = build_model()
model_laplace.fit(train_gen_laplace, validation_data=val_gen, epochs=10)

print("Normal  test acc:", model_normal.evaluate(test_gen)[1])

test_gen.reset()
print("Laplace test acc:", model_laplace.evaluate(test_gen)[1])

## Reconstruction Attack

**Step 1:** Extract feature embeddings (Flatten layer = 36,992-dim vector)

**Step 2:** Build decoder: 36,992-dim → 150x150 image

**Step 3:** Train decoder on (embedding, original_image) pairs

**Step 4:** Measure how well the attacker can reconstruct images

In [ ]:
# ================================
# 7. Extract embeddings from Flatten layer
# ================================

# Sub-model: input → Flatten layer output
encoder_normal  = Model(inputs=model_normal.input,
                        outputs=model_normal.get_layer('flatten').output)
encoder_laplace = Model(inputs=model_laplace.input,
                        outputs=model_laplace.get_layer('flatten').output)

# Collect all test images and their embeddings
test_gen.reset()
all_images = []
all_embeddings_normal  = []
all_embeddings_laplace = []

for i in range(len(test_gen)):
    batch_imgs, _ = test_gen[i]
    all_images.append(batch_imgs)
    all_embeddings_normal.append(encoder_normal.predict(batch_imgs, verbose=0))
    # For Laplace: add noise to images before encoding (simulating DP mechanism)
    batch_noisy = np.array([add_laplace_noise(img) for img in batch_imgs])
    all_embeddings_laplace.append(encoder_laplace.predict(batch_noisy, verbose=0))

X_images           = np.concatenate(all_images, axis=0)           # (N, 150, 150, 1)
Z_normal           = np.concatenate(all_embeddings_normal, axis=0) # (N, 36992)
Z_laplace          = np.concatenate(all_embeddings_laplace, axis=0)# (N, 36992)

print("Images shape:     ", X_images.shape)
print("Embeddings (normal):", Z_normal.shape)
print("Embeddings (laplace):", Z_laplace.shape)

In [ ]:
# ================================
# 8. Build Decoder Model
#    36,992 → 17x17x128 → upsample → 150x150x1
# ================================
def build_decoder():
    inp = Input(shape=(EMBED_DIM,))

    # Reshape back to conv feature map shape: 17x17x128
    x = layers.Reshape((17, 17, 128))(inp)

    # Upsample: 17 → 34
    x = layers.Conv2DTranspose(64, (3,3), strides=2, padding='same', activation='relu')(x)

    # Upsample: 34 → 68
    x = layers.Conv2DTranspose(32, (3,3), strides=2, padding='same', activation='relu')(x)

    # Upsample: 68 → 136
    x = layers.Conv2DTranspose(16, (3,3), strides=2, padding='same', activation='relu')(x)

    # Resize 136 → 150
    x = layers.Resizing(IMG_SIZE, IMG_SIZE)(x)

    # Final output: 150x150x1, pixel values in [0,1]
    x = layers.Conv2D(1, (3,3), padding='same', activation='sigmoid')(x)

    return Model(inp, x, name='decoder')

decoder = build_decoder()
decoder.summary()

In [ ]:
# ================================
# 9. Train decoder on NORMAL embeddings (no privacy)
# ================================
decoder_normal = build_decoder()
decoder_normal.compile(optimizer='adam', loss='mse')

decoder_normal.fit(
    Z_normal, X_images,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# ================================
# 10. Train decoder on LAPLACE embeddings (DP privacy)
# ================================
decoder_laplace = build_decoder()
decoder_laplace.compile(optimizer='adam', loss='mse')

decoder_laplace.fit(
    Z_laplace, X_images,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# ================================
# 11. Compute Reconstruction Metrics
# ================================
recon_normal  = decoder_normal.predict(Z_normal)
recon_laplace = decoder_laplace.predict(Z_laplace)

def compute_metrics(originals, reconstructed, label):
    mse_vals, psnr_vals, ssim_vals = [], [], []

    for orig, recon in zip(originals, reconstructed):
        orig_2d  = orig[:,:,0]
        recon_2d = recon[:,:,0]

        mse_vals.append(np.mean((orig_2d - recon_2d)**2))
        psnr_vals.append(psnr(orig_2d, recon_2d, data_range=1.0))
        ssim_vals.append(ssim(orig_2d, recon_2d, data_range=1.0))

    print(f"\n--- {label} ---")
    print(f"MSE  (↑ better privacy): {np.mean(mse_vals):.4f}")
    print(f"PSNR (↓ better privacy): {np.mean(psnr_vals):.2f} dB")
    print(f"SSIM (↓ better privacy): {np.mean(ssim_vals):.4f}")

    return np.mean(mse_vals), np.mean(psnr_vals), np.mean(ssim_vals)

mse_n, psnr_n, ssim_n = compute_metrics(X_images, recon_normal,  "Baseline (no privacy)")
mse_l, psnr_l, ssim_l = compute_metrics(X_images, recon_laplace, "Differential Privacy (Laplace)")

In [ ]:
# ================================
# 12. Summary Table
# ================================
print("\n" + "="*50)
print(f"{'Metric':<20} {'Baseline':>12} {'DP (Laplace)':>14}")
print("="*50)
print(f"{'MSE (↑=safer)':<20} {mse_n:>12.4f} {mse_l:>14.4f}")
print(f"{'PSNR dB (↓=safer)':<20} {psnr_n:>12.2f} {psnr_l:>14.2f}")
print(f"{'SSIM (↓=safer)':<20} {ssim_n:>12.4f} {ssim_l:>14.4f}")
print("="*50)
print("\nExpected (slide 29): High MSE, Low PSNR, SSIM ≈ 0 → attack fails → privacy holds")

In [ ]:
# ================================
# 13. Visual Comparison
# ================================
n_samples = 4
indices = np.random.choice(len(X_images), n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
fig.suptitle('Reconstruction Attack Results', fontsize=16)

col_titles = ['Original X-ray', 'Reconstructed (No Privacy)', 'Reconstructed (DP Laplace)']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontweight='bold')

for row, idx in enumerate(indices):
    orig  = X_images[idx, :, :, 0]
    r_n   = recon_normal[idx, :, :, 0]
    r_l   = recon_laplace[idx, :, :, 0]

    axes[row, 0].imshow(orig,  cmap='gray', vmin=0, vmax=1)
    axes[row, 1].imshow(r_n,   cmap='gray', vmin=0, vmax=1)
    axes[row, 2].imshow(r_l,   cmap='gray', vmin=0, vmax=1)

    for col in range(3):
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('reconstruction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reconstruction_comparison.png")

In [ ]:
# ================================
# 14. Bar Chart: Metric Comparison
# ================================
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

labels   = ['Baseline', 'DP (Laplace)']
colors   = ['#e74c3c', '#2ecc71']

axes[0].bar(labels, [mse_n, mse_l],   color=colors)
axes[0].set_title('MSE (higher = better privacy)')
axes[0].set_ylabel('MSE')

axes[1].bar(labels, [psnr_n, psnr_l], color=colors)
axes[1].set_title('PSNR dB (lower = better privacy)')
axes[1].set_ylabel('PSNR (dB)')

axes[2].bar(labels, [ssim_n, ssim_l], color=colors)
axes[2].set_title('SSIM (lower = better privacy)')
axes[2].set_ylabel('SSIM')

plt.suptitle('Reconstruction Attack: Baseline vs Differential Privacy', fontsize=13)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: metrics_comparison.png")